# Parsing Log Files

Log files are a primary source of information about system behavior. Python regex makes it easy to extract, filter, and analyze log data.

In [1]:
import re

## Sample Log Data

In [2]:
log_lines = """
2024-01-15 08:23:11 INFO  192.168.1.10 - Request processed
2024-01-15 08:23:45 DEBUG 10.0.0.5 - Cache miss for key 'user_42'
2024-01-15 08:24:02 ERROR 172.16.0.8 - Connection refused [ERR-503]
2024-01-15 08:24:18 WARNING 192.168.1.22 - High memory usage
2024-01-15 08:25:00 ERROR 10.0.0.5 - Disk read failure [ERR-500]
2024-01-15 08:25:30 INFO  192.168.1.10 - User login successful
""".strip().splitlines()

## Extracting IP Addresses with `re.findall`

In [3]:
ip_pattern = r"\b(?:\d{1,3}\.){3}\d{1,3}\b"

all_ips = []
for line in log_lines:
    ips = re.findall(ip_pattern, line)
    all_ips.extend(ips)

print("All IPs found:", all_ips)
print("Unique IPs:", sorted(set(all_ips)))

All IPs found: ['192.168.1.10', '10.0.0.5', '172.16.0.8', '192.168.1.22', '10.0.0.5', '192.168.1.10']
Unique IPs: ['10.0.0.5', '172.16.0.8', '192.168.1.10', '192.168.1.22']


## Filtering Log Levels with `re.search`

In [4]:
def filter_by_level(lines, level):
    pattern = re.compile(rf"\b{level}\b")
    return [line for line in lines if pattern.search(line)]

print("ERROR lines:")
for line in filter_by_level(log_lines, "ERROR"):
    print(" ", line)

print("\nWARNING lines:")
for line in filter_by_level(log_lines, "WARNING"):
    print(" ", line)

ERROR lines:
  2024-01-15 08:24:02 ERROR 172.16.0.8 - Connection refused [ERR-503]
  2024-01-15 08:25:00 ERROR 10.0.0.5 - Disk read failure [ERR-500]

WARNING lines:
  2024-01-15 08:24:18 WARNING 192.168.1.22 - High memory usage


## Extracting Error Codes

In [5]:
error_code_pattern = r"\[ERR-(\d+)\]"

for line in log_lines:
    match = re.search(error_code_pattern, line)
    if match:
        print(f"Error code {match.group(1)} in: {line.strip()}")

Error code 503 in: 2024-01-15 08:24:02 ERROR 172.16.0.8 - Connection refused [ERR-503]
Error code 500 in: 2024-01-15 08:25:00 ERROR 10.0.0.5 - Disk read failure [ERR-500]


## Parsing Log Entries into Structured Objects

Use `re.split` to break each log line into components.

In [6]:
LOG_PATTERN = re.compile(
    r"(?P<date>\d{4}-\d{2}-\d{2}) "
    r"(?P<time>\d{2}:\d{2}:\d{2}) "
    r"(?P<level>\w+) "
    r"(?P<ip>[\d.]+) - "
    r"(?P<message>.+)"
)

def split_log_entry(line):
    match = LOG_PATTERN.match(line.strip())
    if match:
        return match.groupdict()
    return None

for line in log_lines:
    entry = split_log_entry(line)
    if entry:
        print(f"[{entry['level']}] {entry['date']} {entry['time']} | IP: {entry['ip']} | {entry['message']}")

[DEBUG] 2024-01-15 08:23:45 | IP: 10.0.0.5 | Cache miss for key 'user_42'
[ERROR] 2024-01-15 08:24:02 | IP: 172.16.0.8 | Connection refused [ERR-503]
[WARNING] 2024-01-15 08:24:18 | IP: 192.168.1.22 | High memory usage
[ERROR] 2024-01-15 08:25:00 | IP: 10.0.0.5 | Disk read failure [ERR-500]


## Live Log Stream Filtering

Simulate filtering a stream of log lines in real time.

In [7]:
def filter_errors(log_stream):
    """Yield only ERROR and WARNING lines from a log stream."""
    pattern = re.compile(r"\b(ERROR|WARNING)\b")
    for line in log_stream:
        if pattern.search(line):
            yield line.strip()

print("Alerts from log stream:")
for alert in filter_errors(log_lines):
    print(" ", alert)

Alerts from log stream:
  2024-01-15 08:24:02 ERROR 172.16.0.8 - Connection refused [ERR-503]
  2024-01-15 08:24:18 WARNING 192.168.1.22 - High memory usage
  2024-01-15 08:25:00 ERROR 10.0.0.5 - Disk read failure [ERR-500]


## Counting Events by Level

In [8]:
from collections import Counter

level_pattern = re.compile(r"\b(DEBUG|INFO|WARNING|ERROR)\b")

levels = []
for line in log_lines:
    match = level_pattern.search(line)
    if match:
        levels.append(match.group(1))

counts = Counter(levels)
for level, count in sorted(counts.items()):
    print(f"{level:<10} {count}")

DEBUG      1
ERROR      2
INFO       2
WARNING    1


## Processing a Text Log File into Objects

In [9]:
import tempfile, os

# Write log to temp file
with tempfile.NamedTemporaryFile(mode='w', suffix='.log', delete=False) as f:
    f.write("\n".join(log_lines))
    log_path = f.name

# Read and parse
parsed_entries = []
with open(log_path) as f:
    for line in f:
        entry = split_log_entry(line)
        if entry:
            parsed_entries.append(entry)

print(f"Parsed {len(parsed_entries)} log entries")
print("First entry:", parsed_entries[0])

os.unlink(log_path)

Parsed 4 log entries
First entry: {'date': '2024-01-15', 'time': '08:23:45', 'level': 'DEBUG', 'ip': '10.0.0.5', 'message': "Cache miss for key 'user_42'"}
